In [ ]:
!pip install transformers

^C


Defaulting to user installation because normal site-packages is not writeable
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.6 MB 3.5 MB/s eta 0:00:04
   ------ --------------------------------- 1.8/11.6 MB 3.5 MB/s eta 0:00:03
   --------- ------------------------------ 2.6/11.6 MB 3.5 MB/s eta 0:00:03
   ----------- ---------------------------- 3.4/11.6 MB 3.5 MB/s eta 0:00:03
   -------------- ------------------------- 4.2/11.6 MB 3.5 MB/s eta 0:00:03
   ---------------- ----------------------- 4.7/11.6 MB 3.5 MB/s eta 0:00:02
   ------------------ --------------------- 5.5/11.6 MB 3.5 MB/s eta 0:00:02
   --------------------- ------------------ 6.3/11.6 MB 3.5 MB/s eta 0:00:02
   ------------------------ --------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import joblib

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Model_Loaded_Weights/Lung_Model") #Lung_Model path #change the path inside

# Load model
model = AutoModelForSequenceClassification.from_pretrained("Model_Loaded_Weights/Lung_Model") #Lung_Model path # change the path inside

#Load encoder
encoder = joblib.load("Model_Loaded_Weights/Lung_Model/label_encoder.pkl") # label_encoder.pk1 path #Change the path inside

# Move model to GPU (if available)
#model_loaded.to(device)

#model_loaded.eval()

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

In [ ]:
import torch

def predict_disease_k(model, tokenizer, texts, encoder=None, max_length=64, top_k=3):
    """
    Predict diseases from symptom text.

    Parameters
    ----------
    model : AutoModelForSequenceClassification
    tokenizer : AutoTokenizer
    texts : str or list[str]
    encoder : LabelEncoder (optional)
    max_length : int
    top_k : int

    Returns
    -------
    list[dict]
    """

    if isinstance(texts, str):
        texts = [texts]

    device = next(model.parameters()).device
    model.eval()

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)

    results = []

    for i, probs in enumerate(probabilities):

        probs_cpu = probs.cpu()

        top_probs, top_indices = torch.topk(probs_cpu, top_k)

        prediction = {}

        prediction["input"] = texts[i]

        prediction["predicted_class_id"] = int(top_indices[0])

        prediction["confidence"] = float(top_probs[0])

        if encoder is not None:
            prediction["predicted_disease"] = encoder.inverse_transform(
                [int(top_indices[0])]
            )[0]
        else:
            prediction["predicted_disease"] = int(top_indices[0])

        prediction["top_predictions"] = []

        for idx, prob in zip(top_indices, top_probs):

            idx = int(idx)
            prob = float(prob)

            if encoder is not None:
                disease = encoder.inverse_transform([idx])[0]
            else:
                disease = idx

            prediction["top_predictions"].append({
                "class_id": idx,
                "disease": disease,
                "probability": prob
            })

        results.append(prediction)

    return results

In [ ]:
predict_disease_k(
    model,
    tokenizer,
    [
        " Can't breath well w"
    ],
    encoder
)

[{'input': " Can't breath well when laughing hard",
  'predicted_class_id': 15,
  'confidence': 0.501664936542511,
  'predicted_disease': 'bronchitis',
  'top_predictions': [{'class_id': 15,
    'disease': 'bronchitis',
    'probability': 0.501664936542511},
   {'class_id': 4,
    'disease': 'Bronchiectasis',
    'probability': 0.20239876210689545},
   {'class_id': 14,
    'disease': 'bronchiolitis',
    'probability': 0.14737698435783386}]}]